# Module 13 — Tool Calling + API Agents
Colab-ready lab: typed tools → validation → authorization → approval → idempotency → failure injection.

In [ ]:
from dataclasses import dataclass
from enum import Enum
class Risk(Enum): READ='read'; WRITE='write'; HIGH='high'
@dataclass
class Tool: name:str; fn:object; role:str; risk:Risk; idempotent:bool=True
tools={}
def register(t): tools[t.name]=t
register(Tool('get_order',lambda order_id:f'order={order_id}','support',Risk.READ))
register(Tool('create_refund',lambda amount:f'refund={amount}','finance',Risk.HIGH))
print(list(tools))

In [ ]:
completed={}
def call(name,role,args=None,key=None,approved=False):
    t=tools[name]
    if role!=t.role: raise PermissionError('POLICY_DENIED')
    if t.risk is Risk.HIGH and not approved: raise PermissionError('APPROVAL_REQUIRED')
    if key in completed:return completed[key]
    result=t.fn(**(args or {}))
    if key:completed[key]=result
    return result
print(call('get_order','support',{'order_id':'O-1'}))

In [ ]:
try: call('create_refund','finance',{'amount':100},key='r1')
except Exception as e: print(e)
print(call('create_refund','finance',{'amount':100},key='r1',approved=True))
print(call('create_refund','finance',{'amount':100},key='r1',approved=True))

## Detailed exercises
1. Add JSON-schema-style argument validation.
2. Add five tools with different risk classes.
3. Add RBAC roles.
4. Add approval persistence.
5. Add idempotency.
6. Inject timeout and retryable errors.
7. Inject malicious tool output and prove it cannot override policy.
8. Add tool-call budgets.
9. Add structured audit events.
10. Build a mini customer-support Data Agent.
11. Generate 100 tool calls and calculate success/error/denial rates.
12. Design tool version compatibility.

## Failure injection
Unknown tool; malformed arguments; unauthorized role; approval bypass attempt; duplicate delivery; timeout after side effect; malicious tool result; excessive tool loop; oversized tool result.

**Gold challenge:** build a finance agent where the model can propose transfers/refunds but deterministic code controls authorization, approval, budget, idempotency and verification.